# Pay for Content — Browser Use Case

## Overview

In this use case we will build a Strands agent that autonomously pays for and retrieves
content sitting behind a human-facing paywall, using **Amazon Bedrock AgentCore payments**
and the **AgentCore Browser Tool**. The agent launches a managed cloud browser, navigates
to a paywalled page, reads the embedded x402 payment requirement from the DOM, generates
a cryptographic proof via AgentCore payments, interacts with the paywall UI, and returns
the unlocked content — all without any human involvement in the payment step.

### Use Case Details

| Information        | Details                                                       |
|:-------------------|:--------------------------------------------------------------|
| Use case type      | Agentic browser automation with autonomous micropayment       |
| Agent type         | Single                                                        |
| Payment protocol   | x402 (HTTP 402 Payment Required)                              |
| Agentic Framework  | Strands Agents                                                |
| LLM model          | Anthropic Claude Sonnet 4.6                                   |
| Complexity         | Intermediate                                                  |
| SDK used           | boto3 + AgentCore SDK + AgentCorePaymentsPlugin (Strands)     |
| Wallet type        | Embedded crypto wallet (AgentCore-provisioned, Coinbase CDP)  |
| Network            | Base Sepolia testnet (`eip155:84532`); Solana Devnet available |

### Architecture

The agent uses the AgentCore Browser Tool (a managed cloud Chromium session) rather
than a local browser. The paywall page loads normally (HTTP 200) and renders a visible
payment UI. The agent reads the x402 requirement from an embedded `<script>` element,
calls `ProcessPayment` to generate a USDC proof on Base Sepolia, then submits that proof
through the page's JavaScript payment handler. Once the x402 facilitator verifies the
on-chain transaction, the content unlocks and the agent extracts it.

<div style="text-align:left">
    <img src="images/architecture_browser_paywall.png" width="75%"/>
</div>

### Use Case Key Features

* End-to-end agentic browser automation with real on-chain micropayments
* AgentCore Browser Tool provides a managed cloud Chromium session — no local browser required
* Agent never holds private keys; payment signing is delegated to AgentCore payments
* Human-controlled payment limits via `maxSpendAmount` on the payment session
* IAM role separation: `ManagementRole` creates sessions; `ProcessPaymentRole` signs payments
* x402 v2 protocol with `PAYMENT-SIGNATURE` header and USDC on Base Sepolia testnet
* Provider-agnostic design — swap wallet provider config without changing agent logic

## Prerequisites

To execute this use case you will need:
* Python 3.10+
* AWS credentials configured (`aws configure`)
* Amazon Bedrock AgentCore access
* Coinbase CDP account — `CDP_API_KEY_NAME`, `CDP_API_KEY_PRIVATE_KEY`, and `CDP_WALLET_SECRET` from the CDP dashboard
  (this example uses Coinbase CDP; StripePrivy is also supported — swap the credential provider config in Step 3a)
  > **Required:** Enable **Delegated Signing** in your CDP project before running the agent.
  > Go to [portal.cdp.coinbase.com](https://portal.cdp.coinbase.com) → your project → **Wallet** → **Embedded Wallets** → **Policies** → enable **Delegated signing**.
  > Without this, `ProcessPayment` will fail with a delegated signing error.
* IAM roles created — run `bash setup_roles.sh` once if not already done

  > **Content provider:** Deploy the CDK stack before running the agent: `cd content-provider && PAY_TO=0x<your-wallet> bash deploy.sh`. Set `CONTENT_DISTRIBUTION_URL` in `.env` to the printed CloudFront URL.
  > `AgentCoreBrowser` is a cloud-managed browser — it cannot reach `localhost`.

In [ ]:
!pip install -r requirements.txt --quiet

## Step 1 — Configure Your Environment

We load all required values from a `.env` file. Copy `.env.sample` to `.env` and fill
in the values. Step 3 will populate MANAGER_ARN, PAYMENT_CONNECTOR_ID, and PAYMENT_INSTRUMENT_ID on first run.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

REGION = os.environ.get("AWS_REGION", "us-west-2")

# ── Endpoints ────────────────────────────────────────────────────────────────
# CP: credential provider, manager + connector setup. Built from REGION; override CP_ENDPOINT to point at a different host.
CP_ENDPOINT    = os.environ.get("CP_ENDPOINT",
    f"https://bedrock-agentcore-control.{REGION}.amazonaws.com")
# DP: instrument, session, process payment
DP_ENDPOINT    = os.environ.get("DP_ENDPOINT",
    f"https://bedrock-agentcore.{REGION}.amazonaws.com")

# ── Coinbase CDP credentials ──────────────────────────────────────────────────
CDP_API_KEY_NAME        = os.environ["CDP_API_KEY_NAME"]
CDP_API_KEY_PRIVATE_KEY = os.environ["CDP_API_KEY_PRIVATE_KEY"]
CDP_WALLET_SECRET       = os.environ["CDP_WALLET_SECRET"]

# ── Embedded wallet identity ───────────────────────────────────────────────────
WALLET_EMAIL  = os.environ.get("WALLET_EMAIL", "")   # email linkedAccount for embedded wallet

# ── IAM roles ────────────────────────────────────────────────────────────────
MANAGEMENT_ROLE_ARN         = os.environ["MANAGEMENT_ROLE_ARN"]
PROCESS_PAYMENT_ROLE_ARN    = os.environ["PROCESS_PAYMENT_ROLE_ARN"]
CONTROL_PLANE_ROLE_ARN      = os.environ["CONTROL_PLANE_ROLE_ARN"]
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ["RESOURCE_RETRIEVAL_ROLE_ARN"]

# ── Provisioned resource IDs (populated by Step 3 — skip provisioning on re-runs) ──
MANAGER_ARN           = os.environ.get("MANAGER_ARN", "")
PAYMENT_CONNECTOR_ID  = os.environ.get("PAYMENT_CONNECTOR_ID", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")
SESSION_ID            = os.environ.get("SESSION_ID", "")

# ── Session config ────────────────────────────────────────────────────────────
USER_ID                = os.environ.get("USER_ID", "test-user-12345")
SESSION_MAX_SPEND      = os.environ.get("SESSION_MAX_SPEND", "1.00")
SESSION_EXPIRY_MINUTES = int(os.environ.get("SESSION_EXPIRY_MINUTES", "60"))

# ── Network / blockchain ───────────────────────────────────────────────────────
# base-sepolia:  eip155:84532                              (default, e2e tested)
# solana-devnet: solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1  (placeholder, not yet tested)
NETWORK_ALIAS = os.environ.get("NETWORK", "base-sepolia")
NETWORK_MAP = {
    "base-sepolia": {
        "caip2":        "eip155:84532",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
    },
    "solana-devnet": {
        "caip2":        "solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1",
        "botocore_net": "SOLANA",
        "usdc_address": "4zMMC9srt5Ri5X14GAgXhaHii3GnPAEERYPJgZJDncDU",
    },
    "base-mainnet": {
        "caip2":        "eip155:8453",
        "botocore_net": "ETHEREUM",
        "usdc_address": "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913",
    },
}
if NETWORK_ALIAS not in NETWORK_MAP:
    raise ValueError(f"Unknown NETWORK '{NETWORK_ALIAS}'. Valid: {list(NETWORK_MAP)}")
ACTIVE_NETWORK = NETWORK_MAP[NETWORK_ALIAS]

# ── Content provider ──────────────────────────────────────────────────────────
CONTENT_DISTRIBUTION_URL = os.environ.get(
    "CONTENT_DISTRIBUTION_URL",
    "",
)
PAYWALL_DEMO_URL = f"{CONTENT_DISTRIBUTION_URL}/article/paywall-demo"

print(f"✅ Region:        {REGION}")
print(f"✅ CP:            {CP_ENDPOINT}")
print(f"✅ DP:            {DP_ENDPOINT}")
print(f"✅ Network:       {NETWORK_ALIAS} ({ACTIVE_NETWORK['caip2']})")
print(f"✅ Content URL:   {CONTENT_DISTRIBUTION_URL}")
print(f"✅ Payment limit: ${SESSION_MAX_SPEND} USD")
if MANAGER_ARN:
    print(f"✅ Manager ARN:   {MANAGER_ARN} (loaded from .env — Step 3 will be skipped)")
if SESSION_ID:
    print(f"✅ Session ID:    {SESSION_ID} (loaded from .env — Step 4 will be skipped)")

## Step 2 — Initialize AWS Clients

Two clients are required, both using the bundled botocore service models:

| Client | Endpoint | Service name | Used for |
|:-------|:---------|:-------------|:---------|
| `cp_client` | CP (`bedrock-agentcore-control`) | `bedrock-agentcore-control` | `CreatePaymentCredentialProvider`, `CreatePaymentManager`, `CreatePaymentConnector` |
| `mgmt_client` | DP (`bedrock-agentcore`) | `bedrock-agentcore` | `CreatePaymentInstrument`, `CreatePaymentSession`, `GetPaymentSession` |

**IAM role separation:**
* `ControlPlaneRole` — credential provider, payment manager, payment connection (API: `create_payment_connector`) (control plane)
* `ManagementRole` — creates and manages payment sessions. Explicit Deny on `ProcessPayment`
* `ProcessPaymentRole` — agent runtime only. Can only call `ProcessPayment`

In [ ]:
import os
import boto3
from boto3.session import Session
from datetime import datetime

base_session = Session(region_name=REGION)
sts = base_session.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]
print(f"✅ AWS account: {ACCOUNT_ID}")


def assume_role(role_arn: str, session_name: str) -> Session:
    creds = sts.assume_role(RoleArn=role_arn, RoleSessionName=session_name)["Credentials"]
    sess = Session(
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"],
        region_name=REGION,
    )
    assumed_arn = sess.client("sts").get_caller_identity()["Arn"]
    print(f"  → {assumed_arn}")
    return sess


# ── Control plane client (ControlPlaneRole) ───────────────────────────────────
print("\nAssuming ControlPlaneRole...")
cp_session = assume_role(CONTROL_PLANE_ROLE_ARN, f"cp-setup-{int(datetime.now().timestamp())}")
cp_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CP_ENDPOINT)
print("✅ CP client ready (CreatePaymentCredentialProvider / CreatePaymentManager / CreatePaymentConnector)")

# ── Management client (ManagementRole) — sessions ────────────────────────────
print("\nAssuming ManagementRole...")
mgmt_session = assume_role(MANAGEMENT_ROLE_ARN, f"payments-mgmt-{int(datetime.now().timestamp())}")
mgmt_client = mgmt_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)
print("✅ Management client ready (CreatePaymentSession / GetPaymentSession)")

# ── Agent client (ProcessPaymentRole) — runtime only ─────────────────────────
print("\nAssuming ProcessPaymentRole...")
agent_session = assume_role(PROCESS_PAYMENT_ROLE_ARN, f"payments-agent-{int(datetime.now().timestamp())}")
agent_dp_client = agent_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)
print("✅ Agent client ready (ProcessPayment only)")

## Step 3 — Provision Embedded Wallet Resources

These cells run once per user to create the AgentCore payments resource stack:
**CredentialProvider → PaymentManager → PaymentConnection → EmbeddedCryptoWallet Instrument**

If you already have `MANAGER_ARN`, `PAYMENT_CONNECTOR_ID`, and `PAYMENT_INSTRUMENT_ID`
from a previous run, set them in `.env` and skip to Step 4.

> **Embedded wallet:** AgentCore provisions the on-chain wallet — no pre-existing CDP
> wallet is required. The `linkedAccounts` email field ties the wallet to a user identity.
> Coinbase embedded wallets are provisioned synchronously (no OTP/email verification step).
> OTP verification is only required for StripePrivy embedded wallets.

In [ ]:
import uuid
import time

# ── 3a. Create Credential Provider ────────────────────────────────────────────
# The credential provider stores your Coinbase CDP API keys securely.
# For StripePrivy: set credentialProviderVendor="StripePrivy" and
# replace coinbaseCdpConfiguration with stripePlatformConfiguration.
#
# IMPORTANT (Coinbase CDP): Delegated Signing must be enabled in your CDP project
# before ProcessPayment will succeed. Go to portal.cdp.coinbase.com → your project → Wallet → Embedded Wallets → Policies → enable Delegated signing.
cred_resp = cp_client.create_payment_credential_provider(
    name=f"CoinbaseCdp{int(time.time())}",
    credentialProviderVendor="CoinbaseCDP",  # or "StripePrivy"
    providerConfigurationInput={
        "coinbaseCdpConfiguration": {
            "apiKeyId":     CDP_API_KEY_NAME,
            "apiKeySecret": CDP_API_KEY_PRIVATE_KEY,
            "walletSecret": CDP_WALLET_SECRET,
        }
    },
)
CREDENTIAL_PROVIDER_ARN = cred_resp["credentialProviderArn"]
print(f"✅ Credential Provider: {CREDENTIAL_PROVIDER_ARN}")

In [ ]:
# ── 3b. Create Payment Manager ────────────────────────────────────────────────
mgr_resp = cp_client.create_payment_manager(
    name=f"PayMgr{int(time.time())}",
    description="AgentCore payments - Pay for Content Browser use case",
    authorizerType="AWS_IAM",
    roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
    clientToken=str(uuid.uuid4()),
)
MANAGER_ARN    = mgr_resp["paymentManagerArn"]
MANAGER_ID     = mgr_resp["paymentManagerId"]
print(f"✅ Payment Manager ARN: {MANAGER_ARN}")
print(f"   Manager ID:          {MANAGER_ID}")

In [ ]:
# ── 3c. Create Payment Connector ─────────────────────────────────────────────
# Links the manager to the credential provider.
# For StripePrivy: set type="StripePrivy" and update credentialProviderConfigurations accordingly.
conn_resp = cp_client.create_payment_connector(
    paymentManagerId=MANAGER_ID,
    name=f"CoinbaseConn{int(time.time())}",
    description="Coinbase CDP connector for embedded wallet",
    type="CoinbaseCDP",  # or "StripePrivy"
    credentialProviderConfigurations=[{
        "coinbaseCDP": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN}
    }],
    clientToken=str(uuid.uuid4()),
)
PAYMENT_CONNECTOR_ID = conn_resp["paymentConnectorId"]
print(f"✅ Payment Connector ID: {PAYMENT_CONNECTOR_ID}")

In [ ]:
# ── 3d. Create Embedded Crypto Wallet Instrument ──────────────────────────────
# EMBEDDED_CRYPTO_WALLET: AgentCore provisions the wallet — no pre-existing CDP
# wallet needed. The linkedAccounts email ties the wallet to a user identity.
linked_accounts = []
if WALLET_EMAIL:
    linked_accounts = [{"email": {"emailAddress": WALLET_EMAIL}}]

inst_resp = mgmt_client.create_payment_instrument(
    paymentManagerArn=MANAGER_ARN,
    paymentConnectorId=PAYMENT_CONNECTOR_ID,
    userId=USER_ID,
    paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
    paymentInstrumentDetails={
        "embeddedCryptoWallet": {
            "network": ACTIVE_NETWORK["botocore_net"],   # "ETHEREUM" or "SOLANA"
            "linkedAccounts": linked_accounts,
        }
    },
    clientToken=str(uuid.uuid4()),
)
instrument = inst_resp["paymentInstrument"]
PAYMENT_INSTRUMENT_ID = instrument["paymentInstrumentId"]
wallet_details = (
    instrument.get("paymentInstrumentDetails", {})
    .get("embeddedCryptoWallet", {})
)
wallet_address = wallet_details.get("walletAddress", "<pending>")
# WalletHub URL — user opens this to fund wallet and grant signing permission
WALLET_HUB_URL = wallet_details.get("redirectUrl", "")

print(f"✅ Payment Instrument ID: {PAYMENT_INSTRUMENT_ID}")
print(f"   Wallet Address:        {wallet_address}")
print(f"   Network:               {ACTIVE_NETWORK['caip2']}")
if WALLET_HUB_URL:
    print(f"   WalletHub URL:         {WALLET_HUB_URL}")
print()
print("📋 Save these values to .env for future runs:")
print(f"   MANAGER_ARN={MANAGER_ARN}")
print(f"   PAYMENT_CONNECTOR_ID={PAYMENT_CONNECTOR_ID}")
print(f"   PAYMENT_INSTRUMENT_ID={PAYMENT_INSTRUMENT_ID}")

### WalletHub — Fund wallet and grant signing permission

AgentCore has provisioned the embedded wallet. Before the agent can make payments,
you must complete setup in **WalletHub**:

1. **Open the WalletHub URL** printed above. Log in with `WALLET_EMAIL`, review the wallet, and click **Grant signing permission**. The agent cannot sign transactions until this step is complete.
2. Fund the wallet using the wallet address printed above:
   - **Base Sepolia:** go to https://faucet.circle.com → select *Base Sepolia* → paste the wallet address
   - **Solana Devnet:** go to https://faucet.circle.com → select *Solana Devnet* → paste the wallet address
3. **If no WalletHub URL was returned and the instrument status is `ACTIVE`**, the wallet is already authorized for signing. Funding via the faucet is still required before payments succeed.

Once funded, press Enter in the cell below to continue.

In [ ]:
input("Press Enter after you've funded the wallet and granted signing permission in WalletHub...")

### Step 3e — Verify Wallet Balance

After funding the wallet via WalletHub and the testnet faucet, verify the USDC balance
before attempting a payment. If the balance shows 0, the faucet transaction may still be
pending — wait a minute and re-run this cell.

This API supports both Coinbase CDP and StripePrivy wallets on Ethereum and Solana networks.

In [ ]:
# Balance check uses ProcessPaymentRole — same role the agent uses for ProcessPayment.
# Required IAM action: bedrock-agentcore:GetPaymentInstrumentBalance on ProcessPaymentRole.
try:
    balance_resp = agent_dp_client.get_payment_instrument_balance(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=USER_ID,
        chain="BASE_SEPOLIA",
        token="USDC",
    )
    token_balance = balance_resp.get("tokenBalance", {})
    if token_balance:
        amount_units = int(token_balance.get("amount", 0))
        decimals = token_balance.get("decimals", 6)
        readable = amount_units / (10 ** decimals)
        print(f"\u2705 Wallet balance: {readable:.6f} {token_balance.get('token', 'USDC')} on {token_balance.get('chain', 'unknown')}")
    else:
        print("\u26a0\ufe0f  Balance returned empty — faucet may still be pending")
    print(f"   Instrument ID: {PAYMENT_INSTRUMENT_ID}")
except Exception as e:
    print(f"\u26a0\ufe0f  GetPaymentInstrumentBalance failed: {e}")
    print("   Ensure bedrock-agentcore:GetPaymentInstrumentBalance is in the ProcessPaymentRole policy.")
    print("   Continue to Step 4 if the wallet is funded.")


## Step 4 — Create a Payment Session

A payment session enforces payment limits and a time-bound authorization for the agent to spend.
The notebook creates the session via `ManagementRole` — the agent only uses
`ProcessPaymentRole` and cannot create or modify sessions.

Set `SESSION_MAX_SPEND` and `SESSION_EXPIRY_MINUTES` in `.env` to control the payment limits.

In [ ]:
import uuid

session_response = mgmt_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    # userId maps to the X-Amzn-Bedrock-AgentCore-Payments-User-Id HTTP header
    userId=USER_ID,
    expiryTimeInMinutes=SESSION_EXPIRY_MINUTES,
    limits={
        "maxSpendAmount": {
            "value": SESSION_MAX_SPEND,  # must be a string
            "currency": "USD",           # USD not USDC — service converts for budget enforcement
        }
    },
    clientToken=str(uuid.uuid4()),
)

payment_session = session_response["paymentSession"]
SESSION_ID = payment_session["paymentSessionId"]

print(f"✅ Payment session created")
print(f"   Session ID:  {SESSION_ID}")
print(f"   Payment limit: ${SESSION_MAX_SPEND} USD")
print(f"   Expires:     {SESSION_EXPIRY_MINUTES} minutes from now")

if "availableLimits" in payment_session:
    available = payment_session["availableLimits"]["availableSpendAmount"]
    print(f"   Available:   {available['value']} {available['currency']}")

> **Note: two valid x402 browser patterns**
>
> The [AgentCore Browser docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-browser.html) show the browser pattern using raw Playwright with HTTP 402 response interception (`page.on("response", ...)` → `PaymentClient.process_payment` → `page.route()` to inject auth headers on retry). That pattern fits API endpoints accessed through a browser.
>
> This notebook uses a different pattern because real consumer paywalls — NYT, FT, Substack, and similar sites — don't return HTTP 402. They render the page normally (HTTP 200), display the article preview, and hide the premium content behind a payment UI. To reflect that, the content provider here returns HTTP 200 with the x402 requirement embedded in the page DOM (the rendered HTML structure) — specifically inside a `<script id="x402-requirement">` tag that the agent reads after the page loads.
>
> Because there's no HTTP 402 on the response, the plugin's auto-intercept hook doesn't fire. Instead, the agent reads the requirement directly from the page and calls `PaymentManager.generate_payment_header` to produce the proof. The underlying `ProcessPayment` API is the same — only the source of the requirement differs.

## Step 5 — Build the Content Browser Agent

We use `AgentCoreBrowser` from `strands_tools.browser` and the `AgentCorePaymentsPlugin`
from the AgentCore payments SDK. `AgentCoreBrowser` wraps the managed cloud Chromium
session, WebSocket connection, and SigV4 headers. `AgentCorePaymentsPlugin` provides
payment query tools and we use its underlying `PaymentManager` for proof generation
via `generate_payment_header`.

The LLM decides how to sequence the tools:

1. **Browse** to the paywall page and read the x402 requirement from the DOM
2. **Call `process_x402_payment`** with the requirement — internally calls
   `PaymentManager.generate_payment_header` which handles `GetPaymentInstrument`,
   `ProcessPayment`, and proof envelope construction
3. **Fill the proof** into the paywall UI and unlock the content
4. **Extract and return** the article text

> **Note:** x402 requirements are read from the DOM (not from an HTTP 402 response),
> so the plugin's auto-intercept hook does not fire. We construct a synthetic
> `{statusCode: 402, body: requirement}` dict to satisfy the SDK's expected input shape.

Separating browsing from payment keeps each tool focused and makes the agent's
reasoning transparent in the trace.


In [ ]:
import json
import uuid

from strands import Agent, tool
from strands_tools.browser import AgentCoreBrowser
from bedrock_agentcore.payments.integrations.strands import (
    AgentCorePaymentsPlugin,
    AgentCorePaymentsPluginConfig,
)
from bedrock_agentcore.payments.manager import PaymentManager

# Wallet provider: Coinbase CDP (embedded wallet). StripePrivy is also supported —
# swap the credential provider config in Step 3 to switch providers.
# The payment tool and agent logic here are provider-agnostic.

# PaymentManager — initialized with the ProcessPaymentRole session and bundled
# service models so generate_payment_header can reach the correct endpoint.
# The plugin's PaymentManager (created in init_agent) uses default boto3 credentials,
# so we create a separate one here for the payment execution path.
payment_manager = PaymentManager(
    payment_manager_arn=MANAGER_ARN,
    region_name=REGION,
    boto3_session=agent_session,
)

# AgentCorePaymentsPlugin — registers payment query tools (GetPaymentInstrument,
# ListPaymentInstruments, GetPaymentSession) with the agent.
# Note: x402 requirements arrive via DOM (not HTTP 402), so the plugin's
# auto-intercept hook won't fire for this use case.
payments_plugin_config = AgentCorePaymentsPluginConfig(
    payment_manager_arn=MANAGER_ARN,
    user_id=USER_ID,
    region=REGION,
    payment_instrument_id=PAYMENT_INSTRUMENT_ID,
    payment_session_id=SESSION_ID,
)
payments_plugin = AgentCorePaymentsPlugin(config=payments_plugin_config)


@tool
def process_x402_payment(requirement_json: str) -> dict:
    """Process an x402 v2 payment requirement and return a signed proof.

    Uses AgentCore payments SDK (generate_payment_header) with the requirement
    from the paywall page and returns the base64-encoded proof envelope.

    Args:
        requirement_json: JSON string of the x402 requirement object read
                          from the <script id="x402-requirement"> DOM element.

    Returns:
        A dict with 'proof_b64' (the base64 proof to paste into the UI),
        'amount_usdc' (float, amount paid), and 'status' from ProcessPayment.
    """
    requirement = json.loads(requirement_json)

    # The SDK consumes the full requirement (with all accepts entries) and
    # selects the matching accept based on the configured network preferences.
    # Read the first accept here only for the human-readable log line.
    first_accept = requirement["accepts"][0]
    amount_units = int(first_accept.get("maxAmountRequired") or first_accept.get("amount", 0))
    amount_usdc = amount_units / 1_000_000
    print(f"   Processing payment: ${amount_usdc:.6f} USDC on {first_accept['network']}")

    # Build a synthetic 402 response dict that generate_payment_header expects.
    # The browser use case reads x402 from DOM, not an HTTP 402 response, so we
    # construct the expected shape here.
    payment_required_request = {
        "statusCode": 402,
        "headers": {},
        "body": requirement,
    }

    header_dict = payment_manager.generate_payment_header(
        user_id=USER_ID,
        payment_instrument_id=PAYMENT_INSTRUMENT_ID,
        payment_session_id=SESSION_ID,
        payment_required_request=payment_required_request,
        client_token=str(uuid.uuid4()),
    )

    # generate_payment_header returns {"PAYMENT-SIGNATURE": "<base64>"} for x402 v2.
    proof_b64 = list(header_dict.values())[0]

    print(f"\u2705 Proof generated \u2014 ${amount_usdc:.6f} USDC")
    return {
        "proof_b64": proof_b64,
        "amount_usdc": amount_usdc,
        "status": "PROOF_GENERATED",
    }


# AgentCoreBrowser \u2014 managed cloud Chromium session (cannot reach localhost).
agent_core_browser = AgentCoreBrowser(region=REGION)

SYSTEM_PROMPT = """\
You are a content retrieval agent with access to Amazon Bedrock AgentCore payments.
You can autonomously browse paywalled websites and pay for premium content using the
x402 micropayment protocol \u2014 without any human involvement in the payment step.

When asked to retrieve content from a URL, follow these steps in order:

1. Use the browser tool to navigate to the URL.
2. Find the <script id="x402-requirement"> element and read its JSON content.
3. Call process_x402_payment with the full JSON text of that element.
4. Use the browser tool to interact with the paywall UI:
   - Discover payment form elements dynamically using button text, input types,
     and aria-labels \u2014 do not rely on hardcoded IDs from any particular site.
   - On the reference sample content provider the IDs are: pay-btn, proof-input,
     verify-btn, content \u2014 but real x402 sites will differ.
5. Wait for the content to become visible, then extract and return it.
6. Report the content retrieved and the amount paid in USDC.

Always be transparent about what you paid and what content you retrieved.
"""

# NOTE: The element IDs in the system prompt (pay-btn, proof-input, verify-btn, content)
# are specific to this sample content-provider. Real x402 sites will have different
# selectors.

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[agent_core_browser.browser, process_x402_payment],
    plugins=[payments_plugin],
    model="us.anthropic.claude-sonnet-4-6",
)

print("\u2705 Agent ready")
print(f"   Tools:   browser (AgentCoreBrowser), process_x402_payment")
print(f"   Plugin:  AgentCorePaymentsPlugin (payment query tools registered)")
print(f"   Model:   us.anthropic.claude-sonnet-4-6")
print(f"   Network: {NETWORK_ALIAS} ({ACTIVE_NETWORK['caip2']})")


## Step 6 — Retrieve Paywalled Content

Invoke the agent with a plain-language task. It will navigate to the paywall demo page,
read the x402 requirement from the DOM, call `process_x402_payment`, interact with the
paywall UI, and return the unlocked article.

The content provider URL is set via `CONTENT_DISTRIBUTION_URL` in `.env`.
Set `CONTENT_DISTRIBUTION_URL` in `.env` to your deployed CloudFront URL before running this cell.

In [ ]:
result = agent(
    f"Please retrieve the premium article from {PAYWALL_DEMO_URL}. "
    f"Pay for it using x402 and give me a summary of what it contains."
)

### Verify the payment was recorded

Check the session to confirm spend was captured in AgentCore payments.

In [ ]:
session_check = mgmt_client.get_payment_session(
    paymentManagerArn=MANAGER_ARN,
    paymentSessionId=SESSION_ID,
    userId=USER_ID,
)

session_data = session_check["paymentSession"]
print(f"✅ Session verified")
print(f"   Session ID:  {session_data['paymentSessionId']}")
print(f"   Payment limit: ${SESSION_MAX_SPEND} USD")
if "availableLimits" in session_data:
    remaining = session_data["availableLimits"]["availableSpendAmount"]
    print(f"   Remaining:   {remaining['value']} {remaining['currency']}")

## Cleanup

The payment session you created in Step 4 has a built-in expiry (60 minutes by default — set via `SESSION_EXPIRY_MINUTES`). Once the expiry time elapses, the session can no longer be used for payments and the agent can no longer spend.

If you want to clean up the resources you created in Step 3, you can delete the payment instrument, payment connector, payment manager, and credential provider via the AWS CLI or boto3.

In [ ]:
# Verify the session's remaining limit and time before expiry.
# The session will automatically stop accepting payments after expiry; no API call required.
import time
from datetime import datetime

session_check = mgmt_client.get_payment_session(
    paymentManagerArn=MANAGER_ARN,
    paymentSessionId=SESSION_ID,
    userId=USER_ID,
)
session_data = session_check["paymentSession"]
print(f"Session ID:    {session_data['paymentSessionId']}")
if "availableLimits" in session_data:
    remaining = session_data["availableLimits"]["availableSpendAmount"]
    print(f"Remaining:     {remaining['value']} {remaining['currency']}")
print(f"\nSession will stop accepting payments at expiry.")

# Congratulations!

You have built an agent that autonomously browses a paywalled website and pays for
content using **Amazon Bedrock AgentCore payments** and the **AgentCore Browser Tool**.

**What we covered:**

* **`AgentCoreBrowser`** from `strands_tools.browser` — managed cloud Chromium session with no local browser required
* **IAM role separation** — `ManagementRole` creates sessions; `ProcessPaymentRole` signs payments; neither can do both
* **x402 v2 flow** — requirement embedded in DOM (`<script id="x402-requirement">`), proof assembled from `ProcessPayment` response, submitted via the paywall UI
* **Payment limits** — `maxSpendAmount` set by the operator before the agent runs; agent cannot raise its own limit
* **Session expiry** — `expiryTimeInMinutes` set at session creation; payments stop automatically once it elapses

**Next steps:**

* Explore the **Pay for Data** use case for the direct API-based x402 pattern — same `ProcessPayment` API without the browser layer
* Deploy the content provider to AWS using the included CDK stack in `content-provider/` for a persistent demo endpoint